# 04 — ADR extraction with scispaCy bc5cdr

```bash
# bootstrap the 200-review hand-annotation file (do this once, then label by hand)
python -m src.adr_extraction --bootstrap-annotations

# run scispaCy over the full corpus
python -m src.adr_extraction --data data/processed/clean.parquet --out results/adr_mentions.parquet
```

Then this notebook compares scispaCy predictions against the hand-labeled
spans (strict span-match P/R/F1).

In [ ]:
from pathlib import Path
import pandas as pd
import sys
sys.path.insert(0, '..')
from src.adr_extraction import load_scispacy, extract_corpus, load_annotations, parse_span_field
from src.evaluate import span_metrics

ANN = Path('../annotations/adr_manual_200.csv')
ann = load_annotations(ANN)
ann.head(3)

In [ ]:
# scispaCy over the 200 annotated reviews
nlp = load_scispacy()
spans_df = extract_corpus(
    ann.rename(columns={'review': 'review'}),
    nlp=nlp,
    keep_labels=('DISEASE',),
)
spans_df.head()

In [ ]:
# group scispaCy spans per review and compute strict F1 against ADR-labeled gold
pred_per_review = (
    spans_df.groupby('review_id')
             .apply(lambda g: {(int(r['start']), int(r['end']), 'ADR') for _, r in g.iterrows()})
             .to_dict()
)

true_spans, pred_spans = [], []
for _, row in ann.iterrows():
    gold = {(s, e, lab) for (s, e, lab) in row['spans'] if lab == 'ADR'}
    pred = pred_per_review.get(int(row['review_id']), set())
    true_spans.append(gold)
    pred_spans.append(pred)

m = span_metrics(pred_spans, true_spans)
m

In [ ]:
# top scispaCy DISEASE terms across the full corpus
ALL = pd.read_parquet('../results/adr_mentions.parquet')
(ALL.assign(text_lc=ALL['text'].str.lower())
    .loc[lambda d: d['label'] == 'DISEASE']
    ['text_lc'].value_counts().head(30))